# Road Anomaly Detection on the RoadAnomaly dataset
**Deep Learning module project**

**Goal.** Given a road-scene image, produce a pixel-level map of *unexpected* objects on or near the road
(animals, rocks, traffic cones, lost cargo, unusual vehicles…). This is the problem studied by
Lis, Nakka, Fua & Salzmann, *Detecting the Unexpected via Image Resynthesis*, ICCV 2019, for which the
RoadAnomaly dataset was released.

**Plan of the notebook**
1. Dataset exploration — structure, annotations, image/mask correspondence, statistics
2. Preprocessing — label remapping, resizing, stratified train/val/test split, augmentation
3. Models, from the simplest to the most advanced — all built from the ANN/CNN building blocks seen in class:
   (1) ANN (MLP) on image patches, (2) classic CNN classifier on the same patches, (3) U-Net CNN from scratch,
   (4) U-Net with ResNet-34 encoder, random init, (5) same with ImageNet weights (transfer learning),
   (6) DeepLabV3+ (transfer learning), (7) convolutional autoencoder (unsupervised, reconstruction error)
4. Evaluation — precision, recall, F1, IoU, Dice (+ AUROC, AP, FPR@95%TPR), bootstrap confidence intervals
5. Results analysis — qualitative predictions, error maps, per-category and per-size analysis, failure cases

Code is split between this notebook (experiments, figures, discussion) and the documented package
`roadanomaly/` (`data.py`, `models.py`, `metrics.py`, `train.py`). Everything runs on CPU or GPU.

---
### ▶ How to run this notebook in Google Colab
1. **Runtime → Change runtime type → GPU (T4)**. Training everything takes ≈15–20 min on a T4 (≈3 h on CPU).
2. Put `RoadAnomaly.zip` in your Google Drive, in `MyDrive/` (or change `DATA_ZIP` below).
3. **Runtime → Run all**. Drive is mounted; checkpoints, figures and result tables are saved in
   `MyDrive/RoadAnomaly_project/`, so a second run reloads the trained models instead of retraining.
   Set `TRAIN = True` to force retraining.


In [ ]:
# ===== 0. Environment set-up (Colab or local Jupyter) =====
import os, sys, zipfile
IN_COLAB = "google.colab" in sys.modules
%pip install -q segmentation-models-pytorch==0.5.0

DATA_ZIP = "/content/drive/MyDrive/RoadAnomaly.zip"       # <- where you put the dataset zip
WORK_DIR = "/content/drive/MyDrive/RoadAnomaly_project"   # <- checkpoints / figures / results go here
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:  # local execution: run next to the data folder
    WORK_DIR = os.environ.get("WORK_DIR", os.getcwd())
os.makedirs(WORK_DIR, exist_ok=True); os.chdir(WORK_DIR)

DATA_ROOT = os.environ.get("ROADANOMALY_ROOT", "/content/RoadAnomaly" if IN_COLAB else "data/RoadAnomaly")
if not os.path.exists(os.path.join(DATA_ROOT, "frame_list.json")):
    if not os.path.exists(DATA_ZIP) and IN_COLAB:   # fallback: upload the zip by hand
        from google.colab import files
        DATA_ZIP = list(files.upload().keys())[0]
    with zipfile.ZipFile(DATA_ZIP) as z:
        z.extractall(os.path.dirname(DATA_ROOT))
print("data:", DATA_ROOT, "| working dir:", os.getcwd())

## Code — data loading and preprocessing
`load_dataset` reads every frame, **remaps the anomaly label 2 → 1**, checks image/mask sizes, and caches a 640×360 copy; `RoadAnomalyDataset` applies augmentation and ImageNet normalisation.

In [ ]:
"""Data loading, preprocessing and splitting for the RoadAnomaly dataset.

Dataset layout (Lis et al., ICCV 2019):
    RoadAnomaly/frame_list.json
    RoadAnomaly/frames/<name>.webp                         RGB image, 1280x720
    RoadAnomaly/frames/<name>.labels/labels_semantic.png   0 = background, 2 = anomaly

IMPORTANT: the anomaly class is encoded with value **2**, not 1. Masks are
remapped to {0, 1} here so that every downstream component works with a
binary target.
"""

import json
import os
import random
import re
from dataclasses import dataclass

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# Working resolution: keeps the 16:9 aspect ratio (approximately) and is divisible by 32,
# which encoder-decoder networks with 5 down-sampling stages require.
IMG_H, IMG_W = 256, 448
# Images are cached at a slightly larger size so random crops/zoom have room to work.
CACHE_H, CACHE_W = 360, 640


def category_of(name: str) -> str:
    """'animals17_E6_near...' -> 'animals', 'lost_and_found03' -> 'lost'"""
    return re.match(r"[a-zA-Z]+", name).group(0)


@dataclass
class Sample:
    name: str
    category: str
    image: np.ndarray  # HxWx3 uint8 (cached size)
    mask: np.ndarray  # HxW uint8 in {0,1}
    image_full: np.ndarray | None = None  # original resolution (for display)
    mask_full: np.ndarray | None = None


def load_dataset(root: str, keep_full: bool = False) -> list[Sample]:
    """Load every frame + binary mask, checking that image/mask correspond."""
    frames = json.load(open(os.path.join(root, "frame_list.json")))
    out = []
    for f in frames:
        stem = f.rsplit(".", 1)[0]
        img = Image.open(os.path.join(root, "frames", f)).convert("RGB")
        lab = Image.open(os.path.join(root, "frames", stem + ".labels", "labels_semantic.png"))
        lab_np = np.array(lab)
        assert img.size == lab.size, f"size mismatch for {f}: {img.size} vs {lab.size}"
        assert set(np.unique(lab_np)) <= {0, 2}, f"unexpected label values in {f}"
        mask_full = (lab_np == 2).astype(np.uint8)
        img_c = np.array(img.resize((CACHE_W, CACHE_H), Image.BILINEAR))
        mask_c = np.array(Image.fromarray(mask_full * 255).resize((CACHE_W, CACHE_H), Image.NEAREST)) // 255
        out.append(Sample(stem, category_of(stem), img_c, mask_c.astype(np.uint8),
                          np.array(img) if keep_full else None, mask_full if keep_full else None))
    return out


def stratified_split(samples, val_frac=0.17, test_frac=0.17, seed=42):
    """Split per category so every subset contains animals, cones, obstacles, ..."""
    rng = random.Random(seed)
    by_cat: dict[str, list[int]] = {}
    for i, s in enumerate(samples):
        by_cat.setdefault(s.category, []).append(i)
    train, val, test = [], [], []
    for cat, idx in sorted(by_cat.items()):
        rng.shuffle(idx)
        n = len(idx)
        n_test = max(1, round(n * test_frac))
        n_val = max(1, round(n * val_frac)) if n >= 3 else 0
        test += idx[:n_test]
        val += idx[n_test:n_test + n_val]
        train += idx[n_test + n_val:]
    return sorted(train), sorted(val), sorted(test)


class RoadAnomalyDataset(Dataset):
    """Returns (image tensor normalised with ImageNet stats, mask tensor 1xHxW float)."""

    def __init__(self, samples, augment=False):
        self.samples = samples
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def _augment(self, img: np.ndarray, mask: np.ndarray):
        H, W = img.shape[:2]
        # random zoom / crop (scale 0.6-1.0 of the cached frame, aspect kept)
        s = random.uniform(0.6, 1.0)
        ch, cw = int(H * s), int(W * s)
        y0, x0 = random.randint(0, H - ch), random.randint(0, W - cw)
        img, mask = img[y0:y0 + ch, x0:x0 + cw], mask[y0:y0 + ch, x0:x0 + cw]
        if random.random() < 0.5:  # horizontal flip (road scenes are left/right symmetric)
            img, mask = img[:, ::-1], mask[:, ::-1]
        img = img.astype(np.float32)
        # photometric jitter: brightness, contrast, saturation
        img = img * random.uniform(0.75, 1.25)
        mean = img.mean()
        img = (img - mean) * random.uniform(0.75, 1.25) + mean
        gray = img.mean(axis=2, keepdims=True)
        img = (img - gray) * random.uniform(0.7, 1.3) + gray
        return np.clip(img, 0, 255).astype(np.uint8), np.ascontiguousarray(mask)

    def __getitem__(self, i):
        s = self.samples[i]
        img, mask = s.image, s.mask
        if self.augment:
            img, mask = self._augment(img, mask)
        img = np.array(Image.fromarray(np.ascontiguousarray(img)).resize((IMG_W, IMG_H), Image.BILINEAR))
        mask = np.array(Image.fromarray(mask * 255).resize((IMG_W, IMG_H), Image.NEAREST)) // 255
        x = (img.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
        return torch.from_numpy(x.transpose(2, 0, 1)), torch.from_numpy(mask[None].astype(np.float32))


def denormalize(x: torch.Tensor) -> np.ndarray:
    """CHW normalised tensor -> HxWx3 float image in [0,1] for plotting."""
    img = x.detach().cpu().numpy().transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(img, 0, 1)

## Code — models
All models of the study: patch ANN, patch CNN, U-Net from scratch, U-Net / DeepLabV3+ on a ResNet-34 encoder (optionally ImageNet-pretrained) and the convolutional autoencoder.

In [ ]:
"""Model zoo used in the project.

1. UNetScratch      - classic U-Net (Ronneberger et al., 2015), randomly initialised.
                     Baseline CNN: tests what can be learnt from 38 images alone.
2. unet_resnet34    - U-Net decoder on an ImageNet-pretrained ResNet-34 encoder (transfer learning).
3. deeplab_resnet34 - DeepLabV3+ (Chen et al., 2018) with the same pretrained encoder:
                     atrous spatial pyramid pooling gives multi-scale context.
4. ConvAutoencoder  - unsupervised/complementary approach: learns to reconstruct *normal* road
                     content; anomalies are expected to be reconstructed poorly.
"""

import os

import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
import torch.nn.functional as F

# ImageNet ResNet-34 weights (timm release, torchvision-compatible key names)
RESNET34_URL = "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/resnet34-43635321.pth"


def _conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
    )


class UNetScratch(nn.Module):
    def __init__(self, base=32, depth=4, dropout=0.2):
        super().__init__()
        chs = [base * 2 ** i for i in range(depth + 1)]  # 32, 64, 128, 256, 512
        self.downs = nn.ModuleList([_conv_block(3, chs[0])] + [_conv_block(chs[i], chs[i + 1]) for i in range(depth)])
        self.ups = nn.ModuleList([nn.ConvTranspose2d(chs[i + 1], chs[i], 2, stride=2) for i in reversed(range(depth))])
        self.dec = nn.ModuleList([_conv_block(chs[i] * 2, chs[i]) for i in reversed(range(depth))])
        self.drop = nn.Dropout2d(dropout)
        self.head = nn.Conv2d(chs[0], 1, 1)

    def forward(self, x):
        skips = []
        for i, d in enumerate(self.downs):
            x = d(x if i == 0 else F.max_pool2d(x, 2))
            skips.append(x)
        x = self.drop(skips.pop())
        for up, dec in zip(self.ups, self.dec):
            x = up(x)
            x = dec(torch.cat([x, skips.pop()], 1))
        return self.head(x)


def load_imagenet_resnet34(encoder: nn.Module, cache_dir="weights"):
    os.makedirs(cache_dir, exist_ok=True)
    sd = torch.hub.load_state_dict_from_url(RESNET34_URL, model_dir=cache_dir, map_location="cpu", progress=False)
    sd = {k: v for k, v in sd.items() if not k.startswith("fc.")}
    own = encoder.state_dict()
    assert set(sd) <= set(own), set(sd) - set(own)
    own.update(sd)
    nn.Module.load_state_dict(encoder, own, strict=True)
    assert torch.equal(encoder.conv1.weight.detach().cpu(), sd["conv1.weight"])  # weights really loaded
    return encoder


def unet_resnet34(pretrained=True):
    m = smp.Unet("resnet34", encoder_weights=None, classes=1, decoder_attention_type="scse")
    if pretrained:
        load_imagenet_resnet34(m.encoder)
    return m


def deeplab_resnet34(pretrained=True):
    m = smp.DeepLabV3Plus("resnet34", encoder_weights=None, classes=1)
    if pretrained:
        load_imagenet_resnet34(m.encoder)
    return m


class ConvAutoencoder(nn.Module):
    """Under-complete convolutional AE: 3x256x448 -> 128x16x28 bottleneck -> 3x256x448."""

    def __init__(self, ch=(32, 64, 128, 128), bottleneck=128):
        super().__init__()
        enc, cin = [], 3
        for c in ch:
            enc += [nn.Conv2d(cin, c, 4, 2, 1), nn.BatchNorm2d(c), nn.LeakyReLU(0.2, inplace=True)]
            cin = c
        enc += [nn.Conv2d(cin, bottleneck, 3, padding=1)]
        self.encoder = nn.Sequential(*enc)
        dec, cin = [], bottleneck
        for c in reversed(ch):
            dec += [nn.ConvTranspose2d(cin, c, 4, 2, 1), nn.BatchNorm2d(c), nn.ReLU(inplace=True)]
            cin = c
        dec += [nn.Conv2d(cin, 3, 3, padding=1)]
        self.decoder = nn.Sequential(*dec)

    def forward(self, x):
        return self.decoder(self.encoder(x))


class PerceptualExtractor(nn.Module):
    """Frozen ImageNet ResNet-34 stem + layer1 + layer2, used to measure the discrepancy
    between an image and its reconstruction in feature space (idea borrowed from the
    discrepancy network of Lis et al., 2019)."""

    def __init__(self):
        super().__init__()
        enc = smp.Unet("resnet34", encoder_weights=None).encoder
        load_imagenet_resnet34(enc)
        self.stem = nn.Sequential(enc.conv1, enc.bn1, enc.relu, enc.maxpool)
        self.l1, self.l2 = enc.layer1, enc.layer2
        self.eval().requires_grad_(False)

    def forward(self, x):
        f1 = self.l1(self.stem(x))
        f2 = self.l2(f1)
        return [f1, f2]


PATCH = 32  # patch size (pixels at 448x256) used by the two "course-level" baselines below


class PatchANN(nn.Module):
    """Fully-connected network (multilayer perceptron) on a flattened 32x32x3 patch.
    Input 3072 -> 512 -> 128 -> 1 (probability that the patch is anomalous).
    It has no notion of spatial neighbourhood: every pixel position is a separate input."""

    def __init__(self, hidden=(512, 128), dropout=0.3):
        super().__init__()
        layers, cin = [nn.Flatten()], 3 * PATCH * PATCH
        for h in hidden:
            layers += [nn.Linear(cin, h), nn.BatchNorm1d(h), nn.ReLU(inplace=True), nn.Dropout(dropout)]
            cin = h
        layers += [nn.Linear(cin, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class PatchCNN(nn.Module):
    """Classic CNN classifier (as seen in class): 3 x [Conv3x3 - BN - ReLU - MaxPool] then Dense layers.
    32x32x3 -> 16x16x32 -> 8x8x64 -> 4x4x128 -> Flatten(2048) -> Dense(128) -> Dense(1).
    Weight sharing + local receptive fields make it translation-equivariant, unlike the ANN."""

    def __init__(self, dropout=0.3):
        super().__init__()

        def block(cin, cout):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True), nn.MaxPool2d(2))

        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(128 * 4 * 4, 128), nn.ReLU(inplace=True),
                                        nn.Dropout(dropout), nn.Linear(128, 1))

    def forward(self, x):
        return self.classifier(self.features(x))


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

## Code — loss and evaluation metrics
BCE + Dice loss; precision, recall, F1, IoU, Dice from the pixel confusion matrix; AUROC / AP / FPR@95%TPR; validation-based threshold selection.

In [ ]:
"""Losses and evaluation metrics for binary anomaly segmentation."""

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve


class BCEDiceLoss(nn.Module):
    """BCE (pixel-wise calibration) + soft Dice (robust to the ~10% foreground imbalance)."""

    def __init__(self, pos_weight=2.0, dice_w=1.0):
        super().__init__()
        self.pos_weight = pos_weight
        self.dice_w = dice_w

    def forward(self, logits, target):
        bce = F.binary_cross_entropy_with_logits(
            logits, target, pos_weight=torch.tensor(self.pos_weight, device=logits.device))
        p = torch.sigmoid(logits)
        inter = (p * target).sum((1, 2, 3))
        dice = 1 - (2 * inter + 1) / (p.sum((1, 2, 3)) + target.sum((1, 2, 3)) + 1)
        return bce + self.dice_w * dice.mean()


def confusion(pred: np.ndarray, gt: np.ndarray):
    pred, gt = pred.astype(bool), gt.astype(bool)
    tp = np.sum(pred & gt); fp = np.sum(pred & ~gt); fn = np.sum(~pred & gt); tn = np.sum(~pred & ~gt)
    return tp, fp, fn, tn


def scores_from_counts(tp, fp, fn, tn, eps=1e-9):
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)  # identical to Dice for binary masks
    iou = tp / (tp + fp + fn + eps)
    return dict(precision=precision, recall=recall, f1=f1, iou=iou, dice=f1)


def evaluate(scores: list[np.ndarray], gts: list[np.ndarray], threshold: float, n_auc_pixels=2_000_000, seed=0):
    """scores: per-image anomaly maps (same size as gt). Returns dataset-level metrics.

    * precision / recall / F1 / IoU / Dice are computed on the pixels of the whole subset
      (global confusion matrix), the convention used in segmentation benchmarks;
    * mIoU_img / mDice_img average the per-image values (every image weighs the same,
      so small anomalies count as much as large ones);
    * AUROC, AP and FPR@95%TPR are threshold-free metrics used in the anomaly-segmentation
      literature (Lis et al. 2019; Chan et al. 2021, SegmentMeIfYouCan).
    """
    TP = FP = FN = TN = 0
    per_img_iou, per_img_dice = [], []
    for s, g in zip(scores, gts):
        tp, fp, fn, tn = confusion(s >= threshold, g)
        TP += tp; FP += fp; FN += fn; TN += tn
        m = scores_from_counts(tp, fp, fn, tn)
        per_img_iou.append(m["iou"]); per_img_dice.append(m["dice"])
    res = scores_from_counts(TP, FP, FN, TN)
    res["mIoU_img"] = float(np.mean(per_img_iou))
    res["mDice_img"] = float(np.mean(per_img_dice))
    # threshold-free metrics on a random pixel subsample (memory)
    s_all = np.concatenate([s.ravel() for s in scores]); g_all = np.concatenate([g.ravel() for g in gts])
    if len(s_all) > n_auc_pixels:
        idx = np.random.default_rng(seed).choice(len(s_all), n_auc_pixels, replace=False)
        s_all, g_all = s_all[idx], g_all[idx]
    res["auroc"] = roc_auc_score(g_all, s_all)
    res["ap"] = average_precision_score(g_all, s_all)
    fpr, tpr, _ = roc_curve(g_all, s_all)
    res["fpr95"] = float(fpr[np.searchsorted(tpr, 0.95)])
    res["threshold"] = threshold
    res["per_image_iou"] = per_img_iou
    return {k: (float(v) if np.isscalar(v) else v) for k, v in res.items()}


def best_f1_threshold(scores, gts, n_steps=99):
    """Choose the decision threshold maximising pixel F1 (used on the VALIDATION set only)."""
    lo = min(float(s.min()) for s in scores); hi = max(float(s.max()) for s in scores)
    best = (-1, 0.5)
    for t in np.linspace(lo, hi, n_steps + 2)[1:-1]:
        TP = FP = FN = 0
        for s, g in zip(scores, gts):
            tp, fp, fn, _ = confusion(s >= t, g); TP += tp; FP += fp; FN += fn
        f1 = 2 * TP / (2 * TP + FP + FN + 1e-9)
        if f1 > best[0]:
            best = (f1, float(t))
    return best[1], best[0]

## Code — training and inference loops
Generic training loop (AdamW, warm-up + cosine schedule, best-validation checkpointing), sliding-window inference for the patch models, and autoencoder training/scoring.

In [ ]:
"""Training / inference loops (device-agnostic: runs on CUDA if available, else CPU)."""

import json
import math
import os
import time

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed=42):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


def _cosine_with_warmup(opt, total, warmup):
    def f(step):
        if step < warmup:
            return (step + 1) / warmup
        return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)


@torch.no_grad()
def val_iou(model, loader):
    model.eval(); tp = fp = fn = 0; loss = 0.0; crit = BCEDiceLoss()
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        lo = model(x); loss += crit(lo, y).item() * len(x)
        p = lo > 0
        tp += (p & (y > 0.5)).sum().item(); fp += (p & (y < 0.5)).sum().item(); fn += (~p & (y > 0.5)).sum().item()
    return tp / (tp + fp + fn + 1e-9), loss / len(loader.dataset)


def train_segmenter(model, train_samples, val_samples, ckpt_path, epochs=60, lr=1e-3, encoder_lr=None,
                    batch_size=4, weight_decay=1e-4, log_every=5):
    """Train a binary segmentation network with BCE+Dice; keep the checkpoint with the best val IoU.

    encoder_lr: if given, parameters under `model.encoder` use this (smaller) learning rate —
    standard practice when fine-tuning a pretrained backbone (discriminative learning rates).
    """
    model.to(DEVICE)
    tl = DataLoader(RoadAnomalyDataset(train_samples, augment=True), batch_size=batch_size, shuffle=True, drop_last=True)
    vl = DataLoader(RoadAnomalyDataset(val_samples), batch_size=batch_size)
    if encoder_lr is not None and hasattr(model, "encoder"):
        enc = list(model.encoder.parameters()); enc_ids = {id(p) for p in enc}
        groups = [{"params": enc, "lr": encoder_lr}, {"params": [p for p in model.parameters() if id(p) not in enc_ids], "lr": lr}]
    else:
        groups = [{"params": model.parameters(), "lr": lr}]
    opt = torch.optim.AdamW(groups, weight_decay=weight_decay)
    sched = _cosine_with_warmup(opt, epochs * len(tl), warmup=2 * len(tl))
    crit = BCEDiceLoss()
    hist = {"epoch": [], "train_loss": [], "val_loss": [], "val_iou": []}
    best, t0 = -1.0, time.time()
    for ep in range(1, epochs + 1):
        model.train(); run = 0.0
        for x, y in tl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            loss = crit(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            run += loss.item() * len(x)
        iou, vloss = val_iou(model, vl)
        hist["epoch"].append(ep); hist["train_loss"].append(run / (len(tl) * batch_size))
        hist["val_loss"].append(vloss); hist["val_iou"].append(iou)
        if iou > best:
            best = iou; torch.save(model.state_dict(), ckpt_path)
        if ep % log_every == 0 or ep == 1:
            print(f"ep {ep:3d} | train {hist['train_loss'][-1]:.4f} | val {vloss:.4f} | val IoU {iou:.4f} "
                  f"| best {best:.4f} | {time.time() - t0:.0f}s", flush=True)
    json.dump(hist, open(ckpt_path.replace(".pt", "_history.json"), "w"))
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    return hist


@torch.no_grad()
def predict_probs(model, samples, out_hw=(720, 1280), tta=True):
    """Anomaly probability maps, upsampled to the original resolution for evaluation."""
    model.to(DEVICE).eval(); out = []
    ds = RoadAnomalyDataset(samples)
    for i in range(len(ds)):
        x = ds[i][0][None].to(DEVICE)
        p = torch.sigmoid(model(x))
        if tta:  # horizontal-flip test-time augmentation
            p = 0.5 * (p + torch.sigmoid(model(x.flip(-1))).flip(-1))
        p = F.interpolate(p, size=out_hw, mode="bilinear", align_corners=False)
        out.append(p[0, 0].cpu().numpy())
    return out


# ----------------------------------------------------------------------------- patch classifiers (ANN / CNN)
# The image (448x256) is cut into a grid of 8x8 "cells" (56x32 cells). For each cell we feed the classifier the
# 32x32 patch centred on it (context around the cell) and the network predicts the fraction of anomalous pixels in
# that cell. Sliding the window with stride 8 gives a coarse anomaly map. Same input for ANN and CNN => fair comparison.
CELL, PAD = 8, 12  # (32 - 8) / 2 = 12 pixels of context on each side


def extract_patches(x, y=None):
    """x: Bx3xHxW -> (B*nh*nw)x3x32x32 patches; y (optional): matching cell-level anomaly fractions."""
    B, C, H, W = x.shape
    xp = F.pad(x, (PAD, PAD, PAD, PAD), mode="reflect")
    p = F.unfold(xp, kernel_size=32, stride=CELL)  # B x (3*32*32) x L
    p = p.transpose(1, 2).reshape(-1, C, 32, 32)
    if y is None:
        return p
    t = F.avg_pool2d(y, CELL).reshape(-1)  # fraction of anomaly pixels in every 8x8 cell
    return p, t


def train_patch_classifier(model, train_samples, val_samples, ckpt_path, epochs=40, lr=1e-3, batch_images=4,
                           n_pos=192, n_neg=192, log_every=5):
    """Trains on balanced mini-batches of patches (anomalous cells are rare, ~10%)."""
    model.to(DEVICE)
    tl = DataLoader(RoadAnomalyDataset(train_samples, augment=True), batch_size=batch_images, shuffle=True, drop_last=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = _cosine_with_warmup(opt, epochs * len(tl), warmup=len(tl))
    hist = {"epoch": [], "train_loss": [], "val_iou": []}; best, t0 = -1.0, time.time()
    for ep in range(1, epochs + 1):
        model.train(); run = 0.0
        for x, y in tl:
            p, t = extract_patches(x, y)
            pos = torch.nonzero(t > 0).flatten(); neg = torch.nonzero(t == 0).flatten()
            sel = torch.cat([pos[torch.randperm(len(pos))[:n_pos]], neg[torch.randperm(len(neg))[:n_neg]]])
            p, t = p[sel].to(DEVICE), t[sel].to(DEVICE)
            loss = F.binary_cross_entropy_with_logits(model(p).squeeze(1), t)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step(); run += loss.item()
        maps = predict_patch_maps(model, val_samples, out_hw=None)
        tp = fp = fn = 0
        for m, s in zip(maps, RoadAnomalyDataset(val_samples)):
            g = s[1][0].numpy() > 0.5; pr = m >= 0.5
            tp += (pr & g).sum(); fp += (pr & ~g).sum(); fn += (~pr & g).sum()
        iou = tp / (tp + fp + fn + 1e-9)
        hist["epoch"].append(ep); hist["train_loss"].append(run / len(tl)); hist["val_iou"].append(float(iou))
        if iou > best:
            best = iou; torch.save(model.state_dict(), ckpt_path)
        if ep % log_every == 0 or ep == 1:
            print(f"ep {ep:3d} | train BCE {hist['train_loss'][-1]:.4f} | val IoU {iou:.4f} | best {best:.4f} | {time.time() - t0:.0f}s", flush=True)
    json.dump(hist, open(ckpt_path.replace(".pt", "_history.json"), "w"))
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    return hist


@torch.no_grad()
def predict_patch_maps(model, samples, out_hw=(720, 1280)):
    """Sliding-window inference: one score per 8x8 cell, upsampled (bilinear) to out_hw (or 256x448 if None)."""
    model.to(DEVICE).eval(); ds = RoadAnomalyDataset(samples); out = []
    for i in range(len(ds)):
        x = ds[i][0][None]
        s = torch.sigmoid(model(extract_patches(x).to(DEVICE))).reshape(1, 1, x.shape[2] // CELL, x.shape[3] // CELL)
        s = F.interpolate(s, size=out_hw or x.shape[-2:], mode="bilinear", align_corners=False)
        out.append(s[0, 0].cpu().numpy())
    return out


# ----------------------------------------------------------------------------- autoencoder
def train_autoencoder(ae, train_samples, ckpt_path, epochs=150, lr=2e-3, batch_size=4, mask_anomalies=True,
                      log_every=10):
    """Reconstruction training (L1). With mask_anomalies=True the loss ignores annotated anomaly
    pixels, so the AE only ever learns to reproduce *normal* content (road, sky, vegetation...).
    Note: the AE never sees the masks as a target - it is not a segmentation model."""
    ae.to(DEVICE)
    tl = DataLoader(RoadAnomalyDataset(train_samples, augment=True), batch_size=batch_size, shuffle=True, drop_last=True)
    opt = torch.optim.AdamW(ae.parameters(), lr=lr, weight_decay=1e-5)
    sched = _cosine_with_warmup(opt, epochs * len(tl), warmup=2 * len(tl))
    hist = {"epoch": [], "train_loss": []}; t0 = time.time()
    for ep in range(1, epochs + 1):
        ae.train(); run = 0.0
        for x, y in tl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            err = (ae(x) - x).abs()
            w = (1 - y) if mask_anomalies else torch.ones_like(y)
            loss = (err * w).sum() / (w.sum() * 3 + 1e-6)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            run += loss.item()
        hist["epoch"].append(ep); hist["train_loss"].append(run / len(tl))
        if ep % log_every == 0 or ep == 1:
            print(f"ep {ep:3d} | recon L1 {hist['train_loss'][-1]:.4f} | {time.time() - t0:.0f}s", flush=True)
    torch.save(ae.state_dict(), ckpt_path)
    json.dump(hist, open(ckpt_path.replace(".pt", "_history.json"), "w"))
    return hist


def _gauss_blur(t, sigma=4.0):
    k = int(2 * round(3 * sigma) + 1); ax = torch.arange(k, device=t.device) - k // 2
    g = torch.exp(-ax.float() ** 2 / (2 * sigma ** 2)); g = g / g.sum()
    t = F.conv2d(F.pad(t, (k // 2, k // 2, 0, 0), mode="reflect"), g.view(1, 1, 1, k))
    return F.conv2d(F.pad(t, (0, 0, k // 2, k // 2), mode="reflect"), g.view(1, 1, k, 1))


@torch.no_grad()
def autoencoder_scores(ae, extractor, samples, out_hw=(720, 1280)):
    """Two anomaly maps per image:
       * 'pixel'      : blurred mean absolute RGB reconstruction error
       * 'perceptual' : 1 - cosine similarity between ResNet features of input and reconstruction
    Also returns the reconstructions for visualisation."""
    ae.to(DEVICE).eval(); extractor.to(DEVICE)
    ds = RoadAnomalyDataset(samples); pix, perc, recs = [], [], []
    for i in range(len(ds)):
        x = ds[i][0][None].to(DEVICE); r = ae(x)
        e = _gauss_blur((r - x).abs().mean(1, keepdim=True))
        fx, fr = extractor(x), extractor(r)
        d = sum(F.interpolate(1 - F.cosine_similarity(a, b, dim=1)[:, None], size=x.shape[-2:], mode="bilinear",
                              align_corners=False) for a, b in zip(fx, fr)) / len(fx)
        d = _gauss_blur(d)
        pix.append(F.interpolate(e, size=out_hw, mode="bilinear")[0, 0].cpu().numpy())
        perc.append(F.interpolate(d, size=out_hw, mode="bilinear")[0, 0].cpu().numpy())
        rec = r[0].cpu().numpy().transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN
        recs.append(np.clip(rec, 0, 1))
    return pix, perc, recs

In [ ]:
import json, time, warnings
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
warnings.filterwarnings("ignore")

TRAIN = False   # False: reuse checkpoints found in WORK_DIR/checkpoints (models without a checkpoint are trained)
                # True : retrain every model from scratch
FIG = "figures"; os.makedirs(FIG, exist_ok=True); os.makedirs("checkpoints", exist_ok=True)
if not torch.cuda.is_available():
    torch.set_num_threads(os.cpu_count())
seed_everything(42)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("device:", DEVICE, "| torch", torch.__version__)

## 1. Dataset exploration
### 1.1 Structure
```
RoadAnomaly/
  frame_list.json                      list of the 60 frame names
  credits.txt                          image sources (web images, various licences)
  frames/<name>.webp                   RGB image
  frames/<name>.labels/labels_semantic.png        per-pixel label: 0 = background, 2 = anomaly
  frames/<name>.labels/labels_semantic_color.png  colour visualisation of the same label
  frames/<name>.labels/labels_instance.png        instance ids (only 46/60 frames)
  frames/<name>.labels/instance_XXX_gc_*          LabelGrab (GrabCut) editor state
```

In [ ]:
frames = json.load(open(os.path.join(DATA_ROOT, "frame_list.json")))
print(len(frames), "frames, e.g.", frames[:3])
lab_dirs = [os.path.join(DATA_ROOT, "frames", f.rsplit(".", 1)[0] + ".labels") for f in frames]
print("frames with labels_semantic.png :", sum(os.path.exists(os.path.join(d, "labels_semantic.png")) for d in lab_dirs))
print("frames with labels_instance.png :", sum(os.path.exists(os.path.join(d, "labels_instance.png")) for d in lab_dirs))

# raw inspection of image sizes and label values (before any preprocessing)
sizes, values = Counter(), Counter()
for f, d in zip(frames, lab_dirs):
    im = Image.open(os.path.join(DATA_ROOT, "frames", f)); lab = np.array(Image.open(os.path.join(d, "labels_semantic.png")))
    sizes[(im.size, lab.shape[::-1])] += 1; values.update(np.unique(lab).tolist())
print("(image size, mask size) ->", dict(sizes))
print("label values present     ->", dict(values), " => anomaly is encoded as 2, NOT 1")

**Image / mask correspondence.** All 60 images and masks are 1280×720 and share the same file stem, so the
pairing is unambiguous. Two pitfalls are worth stating explicitly:
* the anomaly class is stored with value **2** (the value 1 is reserved for "road" in the authors' tooling and
  never appears here). Treating the mask as `mask > 0` or `mask == 2` is correct; `mask == 1` would give an
  empty target and a model that silently learns nothing;
* masks must be resized with **nearest-neighbour** interpolation, otherwise interpolated values appear at the
  borders.

`load_dataset` asserts both points for every frame.

In [ ]:
samples = load_dataset(DATA_ROOT, keep_full=True)
cats = Counter(s.category for s in samples)
frac = np.array([s.mask_full.mean() for s in samples])
inst = []
for s in samples:
    p = os.path.join(DATA_ROOT, "frames", s.name + ".labels", "labels_instance.png")
    inst.append(len(np.unique(np.array(Image.open(p)))) - 1 if os.path.exists(p) else np.nan)
eda = pd.DataFrame({"name": [s.name for s in samples], "category": [s.category for s in samples],
                    "anomaly_frac": frac, "n_instances": inst})
display(eda.groupby("category").agg(n=("name", "size"), mean_anomaly_pct=("anomaly_frac", lambda x: 100 * x.mean()),
                                    min_pct=("anomaly_frac", lambda x: 100 * x.min()),
                                    max_pct=("anomaly_frac", lambda x: 100 * x.max()),
                                    mean_instances=("n_instances", "mean")).round(2))
print(f"anomaly pixels overall: {100 * frac.mean():.2f}%  (median per image {100 * np.median(frac):.2f}%)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
cc = eda.category.value_counts(); ax[0].bar(cc.index, cc.values, color="#4C72B0"); ax[0].set_title("Frames per category")
ax[1].hist(100 * frac, bins=20, color="#DD8452"); ax[1].set_xlabel("% of image covered by anomaly"); ax[1].set_title("Anomaly size distribution")
prior = np.mean([s.mask for s in samples], axis=0)
im = ax[2].imshow(prior, cmap="magma"); ax[2].set_title("Spatial prior: P(anomaly) per pixel"); ax[2].axis("off")
plt.colorbar(im, ax=ax[2], fraction=0.03); plt.tight_layout(); plt.savefig(f"{FIG}/eda_stats.png", bbox_inches="tight"); plt.show()

Observations:
* **Very small dataset** (60 images) dominated by animals (31). Cones, lost cargo and unusual vehicles have
  3–8 examples each — any learned model will see very few examples of these.
* **Strong class imbalance**: anomalies cover ≈10% of pixels on average, but the median is ≈6.6% and some
  objects cover <0.5% of the frame (distant cones, small animals). Pixel accuracy would be meaningless
  (an all-background prediction scores ≈90%), hence the use of IoU/F1/Dice and threshold-free metrics.
* **Spatial prior**: anomalies concentrate in a central horizontal band (on the road, around and below the
  horizon) and almost never touch the top of the frame (sky) or its lateral edges.
  A CNN can exploit this context; it is legitimate for the task but also a possible shortcut — we therefore
  include a "spatial prior only" reference that predicts this average map for every image.
* Annotation semantics: people, cyclists and ordinary cars are background; for animal-drawn carts the whole
  cart + load + animal is labelled anomalous. The "lost" frames are dash-cam video stills with watermarks.
* Images are web photos with heterogeneous viewpoints, weather and quality — very different from the
  car-mounted cameras of Cityscapes.

In [ ]:
def overlay(img, mask, color=(1, 0, 0), alpha=0.45):
    img = img.astype(np.float32) / 255 if img.dtype == np.uint8 else img.copy()
    out = img.copy(); m = mask.astype(bool)
    out[m] = (1 - alpha) * out[m] + alpha * np.array(color)
    return out

show = [eda[eda.category == c].index[0] for c in cats] + list(eda.sort_values("anomaly_frac").index[[0, -1]])
fig, axes = plt.subplots(3, len(show), figsize=(3.2 * len(show), 6.4))
for j, i in enumerate(show):
    s = samples[i]
    color = np.array(Image.open(os.path.join(DATA_ROOT, "frames", s.name + ".labels", "labels_semantic_color.png")).convert("RGB"))
    for r, im_ in enumerate([s.image_full, color, overlay(s.image_full, s.mask_full)]):
        axes[r, j].imshow(im_); axes[r, j].axis("off")
    axes[0, j].set_title(f"{s.category}\n{100 * frac[i]:.1f}% anomaly", fontsize=9)
axes[0, 0].text(-80, 360, "image", rotation=90, va="center"); axes[1, 0].text(-80, 360, "label (colour)", rotation=90, va="center")
axes[2, 0].text(-80, 360, "binary mask overlay", rotation=90, va="center")
plt.tight_layout(); plt.savefig(f"{FIG}/eda_examples.png", bbox_inches="tight"); plt.show()

The overlays confirm that the masks are aligned with the objects (one example per category, plus the
smallest and the largest anomaly). Note that the annotation marks *only* the anomalous objects; the rest
(road, cars, people, buildings) is all "background". People and ordinary cars are **not** anomalies here.

## 2. Preprocessing
* **Label remapping**: `mask = (labels_semantic == 2)` → binary {0,1}.
* **Resizing**: images cached at 640×360, network input 448×256 (16:9 kept, divisible by 32 as required by
  5-stage encoders). Bilinear for images, nearest for masks. Predictions are **upsampled back to 1280×720
  and evaluated against the original full-resolution masks**.
* **Normalisation**: ImageNet mean/std (needed by the pretrained encoders; harmless for the others).
* **Split**: stratified by category (seed 42) so every subset contains animals, cones, obstacles, lost cargo
  and vehicles: 38 train / 11 validation / 11 test. The validation set is used for checkpoint selection and
  threshold choice; the test set is touched **once**, at the end.
* **Augmentation** (train only — vital with 38 images): random zoom-crop (60–100%), horizontal flip,
  brightness/contrast/saturation jitter. No vertical flips or rotations (road geometry has a fixed "up").

In [ ]:
tr_idx, va_idx, te_idx = stratified_split(samples)
train_s, val_s, test_s = ([samples[i] for i in ix] for ix in (tr_idx, va_idx, te_idx))
split = pd.DataFrame({"split": ["train"] * len(tr_idx) + ["val"] * len(va_idx) + ["test"] * len(te_idx),
                      "category": [samples[i].category for i in tr_idx + va_idx + te_idx]})
display(pd.crosstab(split.category, split.split, margins=True)[["train", "val", "test", "All"]])
print("test frames:", [s.name for s in test_s])

ds_aug = RoadAnomalyDataset(train_s[:1], augment=True)
fig, axes = plt.subplots(1, 5, figsize=(17, 2.6))
for k, ax in enumerate(axes):
    x, y = ds_aug[0] if k else RoadAnomalyDataset(train_s[:1])[0]
    ax.imshow(overlay(denormalize(x), y[0].numpy())); ax.axis("off"); ax.set_title("original (resized)" if k == 0 else f"augmentation {k}")
plt.tight_layout(); plt.savefig(f"{FIG}/augmentations.png", bbox_inches="tight"); plt.show()

## 3. Models

### 3.1 From ANN to CNN to segmentation network
The models form a progression; each step adds one idea to the previous one.

| # | Model | Building blocks (course vocabulary) | Why it is included |
|---|---|---|---|
| 1 | **ANN on patches** (1.6 M params) | Flatten → Dense(512) → Dense(128) → Dense(1, sigmoid) | Simplest neural baseline. The image is cut into 8×8-pixel cells; for each cell the ANN sees the surrounding 32×32 patch (flattened to 3072 numbers) and predicts how much of the cell is anomalous. Sliding it over the image gives a coarse anomaly map. |
| 2 | **CNN classifier on patches** (0.36 M params) | 3× [Conv 3×3 → ReLU → MaxPool] → Flatten → Dense(128) → Dense(1) | *Exactly the same input and target as the ANN*: isolates the benefit of convolutions (local receptive fields, weight sharing, translation invariance) with 4.6× fewer parameters. |
| 3 | **U-Net from scratch** (7.8 M) | Conv/ReLU/MaxPool **encoder** (like the CNN above, deeper) + **decoder** of transposed convolutions (learned up-sampling) + **skip connections** | Instead of classifying patches one by one, a *fully convolutional* network (no Dense layer) outputs a full-resolution mask in one pass. Skip connections give back the fine details lost by pooling. |
| 4 | **U-Net, ResNet-34 encoder, random init** | Encoder = ResNet-34, a 34-layer CNN with *residual* (shortcut) connections, which make deep CNNs trainable | Same architecture as 5 but no pretraining → isolates the effect of transfer learning. |
| 5 | **U-Net, ResNet-34, ImageNet weights** (24.6 M) | Same as 4, encoder initialised with weights learnt on 1.2 M ImageNet photos | **Transfer learning**: ImageNet features already encode "animal", "rock", "cone"… — what we need with only 38 training images. Encoder fine-tuned with a 10× smaller learning rate so the pretrained filters are not destroyed. |
| 6 | **DeepLabV3+, ResNet-34, ImageNet** (22.4 M) | Same pretrained CNN encoder; decoder uses *dilated* ("atrous") convolutions = convolutions with holes, which see a wider area without pooling | Alternative segmentation decoder: looks at several scales at once (anomalies range from 0.4% to 42% of the frame). |
| 7 | **Convolutional autoencoder** (1.3 M) | CNN encoder (strided Conv) → small bottleneck → CNN decoder (transposed Conv) | Unsupervised family: trained only to *reconstruct* normal pixels (anomaly pixels are masked out of the loss). At test time, what it fails to reconstruct is flagged as anomalous — same idea as the "resynthesis" of Lis et al. |

**Why no RNN?** RNNs (and LSTM/GRU) process *sequences* — time series, text, video frames over time — by
carrying a hidden state from one step to the next. RoadAnomaly contains 60 independent still photos: there is
no temporal order to exploit, and the useful structure is *spatial* (neighbouring pixels), which is exactly what
convolutions model. An RNN would only become relevant with video (e.g. a CNN per frame + an LSTM over time to
make detections temporally consistent) — mentioned as future work.

### 3.2 Training set-up
**Loss** (models 3–6): BCE (pos_weight 2) + soft Dice — BCE gives well-calibrated per-pixel gradients, Dice counters
the 1:9 class imbalance and directly optimises the overlap metric.
**Optimiser**: AdamW, weight decay 1e-4, 2-epoch warm-up + cosine decay, batch 4. 60 epochs (80 for the
scratch U-Net, which converges more slowly). The checkpoint with the best validation IoU is kept.

**Important framing remark.** Models A–D are *supervised*: they learn what the annotated anomaly classes look
like (the same is true for models 1–2). This is what the assignment asks for and what the 60 pixel-level labels allow, but it differs from the
paper's setting, where the detector is trained on Cityscapes (no anomalies) and RoadAnomaly is used only as a
test set: true anomalies are by definition *unknown* at training time. Model E is closer to that philosophy.
We come back to this in the discussion.

In [ ]:
MODELS = {
    "U-Net (scratch)": ("unet_scratch", lambda: UNetScratch(), dict(epochs=80, lr=1e-3)),
    "U-Net R34 (random init)": ("unet_r34_random", lambda: unet_resnet34(pretrained=False), dict(epochs=60, lr=1e-3)),
    "U-Net R34 (ImageNet)": ("unet_r34_imagenet", lambda: unet_resnet34(pretrained=True), dict(epochs=60, lr=1e-3, encoder_lr=1e-4)),
    "DeepLabV3+ R34 (ImageNet)": ("deeplabv3p_r34_imagenet", lambda: deeplab_resnet34(pretrained=True), dict(epochs=60, lr=1e-3, encoder_lr=1e-4)),
}
nets, hists = {}, {}
# models 1-2: patch classifiers (ANN vs CNN on identical inputs)
PATCH_MODELS = {"ANN (patches)": ("patch_ann", PatchANN), "CNN classifier (patches)": ("patch_cnn", PatchCNN)}
patch_nets = {}
for label, (key, ctor) in PATCH_MODELS.items():
    ckpt = f"checkpoints/{key}.pt"; seed_everything(42); net = ctor()
    if TRAIN or not os.path.exists(ckpt):
        print(f"--- training {label}"); train_patch_classifier(net, train_s, val_s, ckpt, epochs=40)
    net.load_state_dict(torch.load(ckpt, map_location=DEVICE)); patch_nets[label] = net.to(DEVICE).eval()
    hists[label] = json.load(open(ckpt.replace(".pt", "_history.json")))
    print(f"{label:28s} params={count_params(net) / 1e6:5.2f}M  best val IoU={max(hists[label]['val_iou']):.3f} "
          f"(epoch {int(np.argmax(hists[label]['val_iou'])) + 1})")
# models 3-6: segmentation networks
for label, (key, ctor, kw) in MODELS.items():
    ckpt = f"checkpoints/{key}.pt"
    seed_everything(42); net = ctor()
    if TRAIN or not os.path.exists(ckpt):
        print(f"--- training {label}"); train_segmenter(net, train_s, val_s, ckpt, **kw)
    net.load_state_dict(torch.load(ckpt, map_location=DEVICE)); nets[label] = net.to(DEVICE).eval()
    hists[label] = json.load(open(ckpt.replace(".pt", "_history.json")))
    print(f"{label:28s} params={count_params(net) / 1e6:5.1f}M  best val IoU={max(hists[label]['val_iou']):.3f} "
          f"(epoch {int(np.argmax(hists[label]['val_iou'])) + 1})")

ae = ConvAutoencoder()
if TRAIN or not os.path.exists("checkpoints/autoencoder.pt"):
    seed_everything(42); train_autoencoder(ae, train_s, "checkpoints/autoencoder.pt", epochs=150)
ae.load_state_dict(torch.load("checkpoints/autoencoder.pt", map_location=DEVICE)); ae.eval()
ae_hist = json.load(open("checkpoints/autoencoder_history.json"))
print(f"Autoencoder params={count_params(ae) / 1e6:.1f}M  final recon L1={ae_hist['train_loss'][-1]:.4f}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
for label, h in hists.items():
    if "val_loss" in h:
        ax[0].plot(h["epoch"], h["train_loss"], label=label); ax[1].plot(h["epoch"], h["val_loss"], label=label)
    ax[2].plot(h["epoch"], h["val_iou"], label=label, ls="--" if "patches" in label else "-")
ax[0].set_title("segmentation nets: train loss (BCE+Dice)"); ax[1].set_title("segmentation nets: validation loss")
ax[2].set_title("validation IoU (thr 0.5), all supervised models")
for a in ax: a.set_xlabel("epoch")
ax[2].legend(fontsize=8); plt.tight_layout(); plt.savefig(f"{FIG}/learning_curves.png", bbox_inches="tight"); plt.show()

## 4. Evaluation
Protocol:
1. compute anomaly maps at 1280×720 for validation and test images (flip test-time augmentation for A–D);
2. pick, **on the validation set only**, the decision threshold maximising pixel F1 (for the supervised models
   we also report the default 0.5; the autoencoder scores are not probabilities, so a tuned threshold is
   mandatory);
3. report on the test set: precision, recall, F1, IoU, Dice (from the global pixel confusion matrix),
   per-image mean IoU/Dice, and the threshold-free AUROC, AP and FPR@95%TPR.

Note that for binary masks **F1 and Dice are the same quantity** (2TP / (2TP+FP+FN)); IoU = Dice / (2 − Dice).

In [ ]:
gt_val = [s.mask_full for s in val_s]; gt_test = [s.mask_full for s in test_s]
maps_val, maps_test = {}, {}
for label, net in patch_nets.items():
    maps_val[label] = predict_patch_maps(net, val_s); maps_test[label] = predict_patch_maps(net, test_s)
for label, net in nets.items():
    maps_val[label] = predict_probs(net, val_s); maps_test[label] = predict_probs(net, test_s)

extractor = PerceptualExtractor()
pv, qv, _ = autoencoder_scores(ae, extractor, val_s)
pt, qt, recon_test = autoencoder_scores(ae, extractor, test_s)
maps_val["Autoencoder (pixel error)"], maps_test["Autoencoder (pixel error)"] = pv, pt
maps_val["Autoencoder (perceptual error)"], maps_test["Autoencoder (perceptual error)"] = qv, qt

# trivial reference: the spatial prior learnt from training masks (no image content at all)
prior_full = np.array(Image.fromarray((np.mean([s.mask for s in train_s], 0) * 255).astype(np.uint8)).resize((1280, 720), Image.BILINEAR)) / 255.0
maps_val["Spatial prior (no image)"] = [prior_full] * len(val_s); maps_test["Spatial prior (no image)"] = [prior_full] * len(test_s)

rows, results = [], {}
for label in maps_test:
    thr, f1v = best_f1_threshold(maps_val[label], gt_val)
    r = evaluate(maps_test[label], gt_test, thr); results[label] = r
    rows.append({"model": label, "thr (val)": thr, **{k: r[k] for k in ["precision", "recall", "f1", "iou", "dice", "mIoU_img", "mDice_img", "auroc", "ap", "fpr95"]}})
    if label in nets or label in patch_nets:
        r5 = evaluate(maps_test[label], gt_test, 0.5); results[label + " @0.5"] = r5
        rows.append({"model": label + " @0.5", "thr (val)": 0.5, **{k: r5[k] for k in ["precision", "recall", "f1", "iou", "dice", "mIoU_img", "mDice_img", "auroc", "ap", "fpr95"]}})
table = pd.DataFrame(rows).set_index("model")
display(table.style.format("{:.3f}").background_gradient(subset=["f1", "iou", "ap"], cmap="Greens"))
table.to_csv("results_test.csv")

### 4.1 How reliable are these numbers? Bootstrap over test images
The test set has only 11 images, so a single number hides a lot of variance. We resample the test images
with replacement (2000 times) and recompute the global IoU to get 95% confidence intervals.

In [ ]:
def bootstrap_iou(maps, gts, thr, n=2000, seed=0):
    rng = np.random.default_rng(seed)
    c = np.array([[np.sum((m >= thr) & (g > 0)), np.sum((m >= thr) & (g == 0)), np.sum((m < thr) & (g > 0))] for m, g in zip(maps, gts)])
    vals = []
    for _ in range(n):
        tp, fp, fn = c[rng.integers(0, len(c), len(c))].sum(0); vals.append(tp / (tp + fp + fn + 1e-9))
    return np.percentile(vals, [2.5, 97.5])

ci = {}
for label in maps_test:
    lo, hi = bootstrap_iou(maps_test[label], gt_test, results[label]["threshold"]); ci[label] = (lo, hi)
    print(f"{label:32s} IoU {results[label]['iou']:.3f}   95% CI [{lo:.3f}, {hi:.3f}]")

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
rng = np.random.default_rng(0)
g_all = np.concatenate([g.ravel() for g in gt_test]); sub = rng.choice(len(g_all), 1_000_000, replace=False)
for label in maps_test:
    s_all = np.concatenate([m.ravel() for m in maps_test[label]])[sub]
    p, r, _ = precision_recall_curve(g_all[sub], s_all); f, t, _ = roc_curve(g_all[sub], s_all)
    ls = "--" if "Autoencoder" in label or "prior" in label else "-"
    ax[0].plot(r, p, ls, label=f"{label} (AP={results[label]['ap']:.2f})"); ax[1].plot(f, t, ls, label=f"{label} (AUROC={results[label]['auroc']:.2f})")
ax[0].set_xlabel("recall"); ax[0].set_ylabel("precision"); ax[0].set_title("Precision–recall (test pixels)")
ax[1].set_xlabel("false positive rate"); ax[1].set_ylabel("true positive rate"); ax[1].set_title("ROC (test pixels)")
ax[0].legend(fontsize=7); ax[1].legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{FIG}/pr_roc.png", bbox_inches="tight"); plt.show()

## 5. Results analysis
### 5.1 Qualitative predictions
Green = true positive, red = false positive, blue = missed anomaly (false negative).

In [ ]:
def error_map(img, pred, gt):
    out = img.astype(np.float32) / 255 * 0.6
    out[pred & gt] = [0.1, 0.85, 0.1]; out[pred & ~gt] = [0.95, 0.1, 0.1]; out[~pred & gt] = [0.1, 0.35, 1.0]
    return out

show_models = ["ANN (patches)", "CNN classifier (patches)", "U-Net (scratch)", "U-Net R34 (ImageNet)", "DeepLabV3+ R34 (ImageNet)", "Autoencoder (perceptual error)"]
fig, axes = plt.subplots(len(test_s), 2 + len(show_models), figsize=(3.0 * (2 + len(show_models)), 1.75 * len(test_s)))
for i, s in enumerate(test_s):
    axes[i, 0].imshow(s.image_full); axes[i, 1].imshow(overlay(s.image_full, s.mask_full, (0.1, 0.85, 0.1)))
    for j, label in enumerate(show_models):
        pred = maps_test[label][i] >= results[label]["threshold"]; g = s.mask_full.astype(bool)
        iou = (pred & g).sum() / ((pred | g).sum() + 1e-9)
        axes[i, 2 + j].imshow(error_map(s.image_full, pred, g)); axes[i, 2 + j].set_title(f"IoU {iou:.2f}", fontsize=8, pad=2)
    axes[i, 0].set_ylabel(s.category, fontsize=8)
for ax_ in axes.ravel(): ax_.set_xticks([]); ax_.set_yticks([])
for j, t in enumerate(["image", "ground truth"] + show_models):
    axes[0, j].set_title((t + "\n" + axes[0, j].get_title()).strip(), fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIG}/qualitative_test.png", bbox_inches="tight", dpi=90); plt.show()

In [ ]:
best_label = max(nets, key=lambda k: results[k]["iou"])
fig, axes = plt.subplots(4, 4, figsize=(17, 8))
for j, i in enumerate(range(0, len(test_s), max(1, len(test_s) // 4))[:4]):
    s = test_s[i]
    axes[0, j].imshow(s.image_full); axes[0, j].set_title(s.name[:30], fontsize=8)
    axes[1, j].imshow(maps_test[best_label][i], cmap="inferno", vmin=0, vmax=1); axes[1, j].set_title(f"{best_label}: probability", fontsize=8)
    axes[2, j].imshow(recon_test[i]); axes[2, j].set_title("autoencoder reconstruction", fontsize=8)
    axes[3, j].imshow(maps_test["Autoencoder (perceptual error)"][i], cmap="inferno"); axes[3, j].set_title("AE perceptual error", fontsize=8)
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(f"{FIG}/heatmaps_and_reconstructions.png", bbox_inches="tight", dpi=90); plt.show()

### 5.2 Where do the models fail? Per-category and per-size analysis
Test-set per-image IoU for each test frame, against the size of the anomaly.

In [ ]:
per = pd.DataFrame({"frame": [s.name[:28] for s in test_s], "category": [s.category for s in test_s],
                    "anomaly_%": [100 * s.mask_full.mean() for s in test_s]})
for label in list(patch_nets) + list(nets) + ["Autoencoder (perceptual error)"]:
    per[label] = results[label]["per_image_iou"]
display(per.sort_values("anomaly_%").style.format({c: "{:.2f}" for c in per.columns[2:]}).background_gradient(subset=list(per.columns[3:]), cmap="RdYlGn", vmin=0, vmax=1))

fig, ax = plt.subplots(figsize=(7, 4))
for label in ["CNN classifier (patches)"] + list(nets):
    ax.scatter(per["anomaly_%"], per[label], label=label, s=30, alpha=0.8)
ax.set_xscale("log"); ax.set_xlabel("anomaly size (% of image, log)"); ax.set_ylabel("per-image IoU"); ax.legend(fontsize=7)
ax.set_title("Per-image IoU vs. anomaly size (test)"); plt.tight_layout(); plt.savefig(f"{FIG}/iou_vs_size.png", bbox_inches="tight"); plt.show()

### 5.3 Sanity check on the validation set
The threshold and checkpoint were selected on validation, so validation metrics are optimistic — but a large
val→test gap would reveal over-fitting to the selection.

In [ ]:
vrows = []
for label in maps_val:
    thr = results[label]["threshold"]; r = evaluate(maps_val[label], gt_val, thr)
    vrows.append({"model": label, "val IoU": r["iou"], "test IoU": results[label]["iou"], "val AP": r["ap"], "test AP": results[label]["ap"]})
display(pd.DataFrame(vrows).set_index("model").round(3))

json.dump({k: {m: v for m, v in r.items() if m != "per_image_iou"} | {"iou_ci95": list(ci.get(k, (np.nan, np.nan)))}
           for k, r in results.items()}, open("results_test.json", "w"), indent=1, default=float)

## 6. Discussion and conclusions
*(The numbered findings below are written from the outputs above; see the technical report for the full
discussion.)*

1. **ANN vs CNN** — on identical patches, compare rows "ANN (patches)" and "CNN classifier (patches)" in the table above.
2. **Patch classification vs segmentation** — compare the patch CNN with the U-Nets.
3. **Transfer learning** — compare "U-Net R34 (random init)" with "U-Net R34 (ImageNet)".
4. **Supervised vs unsupervised** — compare the segmentation networks with the autoencoder rows.
5. **Limits** — 11 test images (see the bootstrap confidence intervals), supervised models only recognise anomaly *types* seen in training, small and distant objects are the main failure cases.